# AN6 — Recorded VLE Activity Patterns

**Status:** Prepared; no SQL outputs are embedded. Run cells individually, not Run All. This notebook describes recorded platform activity, not learning quality.


## 1. Environment setup

This cell locates the repository and imports the same canonical AN6 execution helpers used by `scripts/run_an6.py`. It does not connect to PostgreSQL.

In [ ]:
# 1. Locate the repository and import the canonical AN6 execution helpers.
from pathlib import Path
import sys
from IPython.display import display

_current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [_current, *_current.parents]
        if (candidate / "src" / "analytics" / "db.py").is_file()
        and (candidate / "sql" / "04_analytics").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root or notebooks folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.run_an6 import (
    AN6_QUERY_IDS,
    EXPECTED_RESULT_SET_COUNTS,
    readiness_gate,
    run_an6_query,
    verify_read_only_preflight,
)

import pandas as pd
import matplotlib.pyplot as plt

query_results = {}
AN6_READINESS_PASSED = False
HIGH_COST_TIMEOUT_MS = 600_000

print("Project root:", PROJECT_ROOT)
print("AN6 query IDs:", AN6_QUERY_IDS)
print("Canonical AN6 runner imported:", callable(run_an6_query))

## 2. AN0.01 — Database identity and read-only preflight

The canonical preflight verifies the intended database and both read-only settings. The returned database identity, session settings, and elapsed time are displayed as execution evidence. No AN6 analytical query runs unless this preflight passes.

In [ ]:
# 2. Verify the intended database and read-only session before AN6 execution.
preflight = verify_read_only_preflight()
query_results["PREFLIGHT.AN0.01"] = preflight
display(preflight["data"])
print("Preflight elapsed seconds:", preflight["elapsed_seconds"])
print("Preflight: PASS — PostgreSQL connection and read-only session confirmed.")

## 3. AN6.01 — Readiness and metadata

**Question:** Are all 22 course windows valid, are course-scoped VLE resource keys unique, and are registration/activity keys complete? Review all three result sets. A failed readiness gate blocks AN6.02–AN6.04.

In [ ]:
# 3. Execute AN6.01, display every result set, and enforce the readiness gate.
r = run_an6_query("AN6.01", max_rows=500)
query_results["AN6.01"] = r

for item in r["result_sets"]:
    print(
        f"Result set {item['statement_number']}: "
        f"{item['rows_displayed']} rows; truncated={item['display_truncated']}"
    )
    display(item["data"])

readiness_gate(r)
AN6_READINESS_PASSED = True
meta = r["result_sets"][1]["data"]
print(
    "Readiness passed. Metadata missingness is descriptive, not a filter:",
    meta[
        [
            "missing_activity_type_rows",
            "missing_week_from_rows",
            "missing_week_to_rows",
            "reversed_week_range_rows",
        ]
    ],
)


## 4. AN6.02 — In-course coverage and student-presentation distributions

**Question:** What share of registered student-presentations has in-course activity? How are zero-inclusive active days and resource breadth distributed? First activity day is conditional on activity being recorded.


In [ ]:
assert AN6_READINESS_PASSED, 'Run and pass the AN6.01 readiness gate before this query.'
r=run_an6_query('AN6.02',max_rows=500,statement_timeout_ms=HIGH_COST_TIMEOUT_MS)
assert r['result_set_count']==EXPECTED['AN6.02']
query_results['AN6.02']=r
an602=r['result_sets'][0]['data']
display(an602)


## 5. AN6.03 — Temporal patterns

**Question:** How do activity rows distribute across PRE/IN/POST and fixed seven-day bins? These bins are not academic weeks.


In [ ]:
assert AN6_READINESS_PASSED, 'Run and pass the AN6.01 readiness gate before this query.'
r=run_an6_query('AN6.03',max_rows=500,statement_timeout_ms=HIGH_COST_TIMEOUT_MS)
assert r['result_set_count']==EXPECTED['AN6.03']
query_results['AN6.03']=r
display(r['result_sets'][0]['data'])


## 6. AN6.04 — Activity-type composition

**Question:** Which resource types contribute to in-course resource breadth and registered student-presentation reach? Missing and unmatched metadata remain explicit.


In [ ]:
assert AN6_READINESS_PASSED, 'Run and pass the AN6.01 readiness gate before this query.'
r=run_an6_query('AN6.04',max_rows=500,statement_timeout_ms=HIGH_COST_TIMEOUT_MS)
assert r['result_set_count']==EXPECTED['AN6.04']
query_results['AN6.04']=r
display(r['result_sets'][0]['data'])


## 7. Descriptive visual summary

Charts are created only from returned SQL frames; no pre-execution figures or findings are embedded.


In [ ]:
if 'AN6.02' in query_results:
    frame=query_results['AN6.02']['result_sets'][0]['data']
    cov=frame[frame.output_section=='COVERAGE'].copy()
    if not cov.empty:
        cov['presentation']=cov.code_module.astype(str)+' '+cov.code_presentation.astype(str)
        ax=cov.plot(x='presentation',y='pct_or_mean',kind='bar',legend=False,figsize=(12,5),title='Recorded in-course activity coverage by presentation')
        ax.set_ylabel('Registered student-presentations with in-course activity (%)')
        plt.tight_layout(); plt.show()
    display(frame[frame.output_section=='DISTRIBUTION'])


## 8. Review checklist

- Verify result-set counts and truncation/runtime metadata.
- Reconcile registration denominators and PRE/IN/POST activity rows.
- Review missing/unmatched VLE metadata.
- Do not interpret missing VLE rows as absence of learning or negative course-relative days as pre-registration activity.
- No composite score, threshold, outcome claim, or predictive feature is approved in AN6.
